# 01 — Cross-Polytope Calibration

Grid search over `num_hashes`, `num_tables`, `num_probes` on the
**validation set only** (protocol section 8). The test set is never
touched here. Parametric: one Kaggle session per split seed.

Method evaluated: **CrossPolytope** (FALCONN-style) only.


In [ ]:
import numpy as np
import pandas as pd
import time
import json
import pathlib
import sys

try:
    from tqdm.notebook import tqdm
except ImportError:
    import tqdm

root_dir = pathlib.Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))


In [ ]:
# ============================================================
# CONFIG — one Kaggle session per (seed, dataset, base_size)
# ============================================================
SPLIT_SEED = 42          # one of: 42, 7, 123, 2024, 99
DATASET = 'heart'        # 'heart' or 'chest'
BASE_SIZE = None         # None=full, 10_000, 50_000 (chest only)

HEART_CSV_PATH = ''
CHEST_EMBEDDINGS_PATH = ''
CHEST_LABELS_PATH = ''
# ============================================================


In [ ]:
from config.seeds import SPLIT_SEEDS, HASH_SEEDS
from config.metrics import K_NEIGHBOURS

assert SPLIT_SEED in SPLIT_SEEDS, f'Seed {SPLIT_SEED} not in {SPLIT_SEEDS}'
HASH_SEED_CALIB = 11     # calibration uses a single hash seed (protocol)


In [ ]:
if DATASET == 'heart':
    from src.datasets.heart_disease import load_and_split
    X_base, y_base, X_val, y_val, X_test, y_test = load_and_split(
        HEART_CSV_PATH, split_seed=SPLIT_SEED, normalise_cosine=True
    )
else:
    from src.datasets.chest_xray14 import load_and_split
    X_base, y_base, X_val, y_val, X_test, y_test = load_and_split(
        CHEST_EMBEDDINGS_PATH, CHEST_LABELS_PATH,
        split_seed=SPLIT_SEED, normalise_cosine=True,
    )
    if BASE_SIZE is not None:
        idx = np.random.default_rng(SPLIT_SEED).choice(
            len(X_base), size=min(BASE_SIZE, len(X_base)), replace=False
        )
        X_base = X_base[idx]
        y_base = y_base[idx]

print(f'X_base: {X_base.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')


In [ ]:
from src.indexes.linear_exact import LinearExact
t0 = time.time()
linear = LinearExact()
linear.build(X_base)
gt_val = []
for x in X_val:
    idxs, _ = linear.query(x, k=K_NEIGHBOURS)
    gt_val.append(idxs)
gt_val = np.array(gt_val)
print(f'Linear ground truth for {len(X_val)} val queries in {time.time()-t0:.1f}s')


In [ ]:
from sklearn.metrics import f1_score

def majority_vote(labels):
    if len(labels) == 0:
        return -1
    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]

lin_preds = [majority_vote(y_base[gt]) for gt in gt_val]
linear_f1 = f1_score(y_val, lin_preds, average='macro', zero_division=0)
print(f'Linear macro-F1 on val: {linear_f1:.4f}')


In [ ]:
from src.indexes.cross_polytope import CrossPolytope
from src.evaluation.metrics import recall_at_k, candidate_fraction

grid = {
    'heart': {'num_hashes': [1, 2], 'num_tables': [5, 10, 20], 'num_rotations': [2], 'num_probes': [1, 4, 8]},
    'chest': {'num_hashes': [1, 2], 'num_tables': [10, 20, 30], 'num_rotations': [2], 'num_probes': [1, 4, 8]},
}
param_grid = grid[DATASET]
keys, values = zip(*param_grid.items())
combos = [dict(zip(keys, v)) for v in __import__('itertools').product(*values)]


In [ ]:
cp_results = []
for p in tqdm(combos, desc='CrossPolytope combos'):
    cp = CrossPolytope(
        num_hashes=p['num_hashes'],
        num_tables=p['num_tables'],
        num_rotations=p['num_rotations'],
        num_probes=p['num_probes'],
        hash_seed=HASH_SEED_CALIB,
    )
    t0 = time.time()
    cp.build(X_base)
    build_time = time.time() - t0

    recalls, latencies, cands, preds = [], [], [], []
    for i, q in enumerate(X_val):
        t0 = time.perf_counter()
        idxs, _, n_cand, _ = cp.query(q, k=K_NEIGHBOURS)
        latencies.append(time.perf_counter() - t0)
        recalls.append(recall_at_k(idxs, gt_val[i], K_NEIGHBOURS))
        cands.append(candidate_fraction(n_cand, len(X_base)))
        preds.append(majority_vote(y_base[idxs]))

    r10 = float(np.mean(recalls))
    f1 = f1_score(y_val, preds, average='macro', zero_division=0)
    f1_drop = linear_f1 - f1
    passed = (r10 >= 0.95) and (f1_drop <= 0.01)
    cp_results.append({
        'num_hashes': p['num_hashes'], 'num_tables': p['num_tables'],
        'num_rotations': p['num_rotations'], 'num_probes': p['num_probes'],
        'hash_seed': HASH_SEED_CALIB,
        'recall@10': r10, 'macro_f1': f1, 'f1_drop': f1_drop,
        'median_latency': float(np.median(latencies)),
        'mean_cand_fraction': float(np.mean(cands)),
        'build_time': build_time, 'passed': passed,
    })

cp_df = pd.DataFrame(cp_results)
cp_df['dataset'] = DATASET
cp_df['base_size'] = str(BASE_SIZE)
cp_df['split_seed'] = SPLIT_SEED
print(cp_df.sort_values(['passed', 'median_latency'], ascending=[False, True]).head(10))


In [ ]:
passing = cp_df[cp_df['passed']] if cp_df['passed'].any() else cp_df
best_cp = passing.sort_values(['median_latency', 'mean_cand_fraction']).iloc[0]
print('=== Best CrossPolytope ===')
print(best_cp[['num_hashes', 'num_tables', 'num_probes', 'num_rotations',
              'recall@10', 'macro_f1', 'median_latency', 'mean_cand_fraction']].to_dict())


In [ ]:
OUTPUT_DIR = pathlib.Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)
cp_df.to_csv(OUTPUT_DIR / f'lsh_calibration_seed{SPLIT_SEED}.csv', index=False)

best_params = {
    'dataset': DATASET,
    'base_size': str(BASE_SIZE),
    'split_seed': SPLIT_SEED,
    'crosspolytope': {
        'num_hashes': int(best_cp['num_hashes']),
        'num_tables': int(best_cp['num_tables']),
        'num_rotations': int(best_cp['num_rotations']),
        'num_probes': int(best_cp['num_probes']),
        'hash_seed': int(best_cp['hash_seed']),
    },
}
with open(OUTPUT_DIR / f'best_params_seed{SPLIT_SEED}.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print(f'Saved calibration to {OUTPUT_DIR}/lsh_calibration_seed{SPLIT_SEED}.csv')
